In [1]:
import os
os.chdir('../')

In [17]:
import torch
import torch.nn.functional as F
import evaluate
import numpy as np
from transformers import PreTrainedModel, PretrainedConfig, PreTrainedTokenizer
from transformers import TrainingArguments, Trainer
from transformers.modeling_outputs import CausalLMOutput
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType

## 1. HF Wrappers
Wrap our custom model and tokenizer so HF Trainer and PEFT work with them.

In [18]:
class GPTConfigHF(PretrainedConfig):
    model_type = "custom_gpt"

    def __init__(self, vocab_size=8192, context_length=512,
                 d_model=768, n_heads=8, n_layers=8,
                 dropout=0.1, qkv_bias=False, **kwargs):
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout
        self.qkv_bias = qkv_bias
        super().__init__(**kwargs)

In [19]:
class LLMHF(PreTrainedModel):
    config_class = GPTConfigHF

    def __init__(self, config):
        super().__init__(config)
        from config.gpt_config import GPTConfig
        from model.llm import LLM
        gpt_cfg = GPTConfig(
            vocab_size=config.vocab_size,
            context_length=config.context_length,
            d_model=config.d_model,
            n_heads=config.n_heads,
            n_layers=config.n_layers,
            dropout=config.dropout,
            qkv_bias=config.qkv_bias,
        )
        self.model = LLM(gpt_cfg)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        logits = self.model(input_ids)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return CausalLMOutput(loss=loss, logits=logits)

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        return {"input_ids": input_ids}

In [27]:
class BPETokenizerHF(PreTrainedTokenizer):
    def __init__(self, bpe_tokenizer, **kwargs):
        self._bpe = bpe_tokenizer
        super().__init__(eos_token="<|endoftext|>", pad_token="<|endoftext|>", **kwargs)

    @property
    def vocab_size(self):
        return len(self._bpe.vocab)

    def get_vocab(self):
        return dict(self._bpe.inverse_vocab)

    def _tokenize(self, text):
        ids = self._bpe.encode(text)
        return [self._bpe.vocab[i] for i in ids]

    def _convert_token_to_id(self, token):
        return self._bpe.inverse_vocab.get(token, 0)

    def _convert_id_to_token(self, index):
        return self._bpe.vocab.get(index, "<unk>")

    def convert_tokens_to_string(self, tokens):
        ids = [self._convert_token_to_id(t) for t in tokens]
        return self._bpe.decode(ids)

    def save_vocabulary(self, save_directory, filename_prefix=None):
        return ()

## 2. Load Model + Apply LoRA

In [28]:
from tokenizer.bpe import BPETokenizer
from training.trainer import load_checkpoint

# tokenizer
bpe = BPETokenizer()
bpe.load("data/vocab.json", "data/merges.json")
tokenizer = BPETokenizerHF(bpe)

# model
hf_config = GPTConfigHF()
model = LLMHF(hf_config)
optimizer = torch.optim.AdamW(model.parameters())
load_checkpoint("checkpoints/ckpt_epoch3.pt", model.model, optimizer, torch.device("cpu"))

[transformers] LLMHF has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Resumed from checkpoints/ckpt_epoch3.pt (epoch 3, step 20742, mid_epoch=False)


(2,
 20742,
 [1.6490518569946289,
  1.6487317740917207,
  1.6542361736297608,
  1.6476067543029784,
  1.6506493806838989,
  1.6368923485279083,
  1.636522376537323,
  1.6250500142574311,
  1.627065360546112,
  1.6123258173465729,
  1.60555180311203,
  1.6117181718349456,
  1.598663228750229,
  1.6062423944473267,
  1.5991171717643737,
  1.5932637929916382,
  1.5955742716789245,
  1.5974591195583343,
  1.5914544105529784,
  1.575858211517334,
  1.5741069197654725,
  1.5802173376083375,
  1.5775056898593902,
  1.5589122951030732,
  1.5659343540668487,
  1.5639585614204408,
  1.5550651133060456,
  1.5550644040107726,
  1.5599053740501403,
  1.5496776580810547,
  1.5489974737167358,
  1.5489375472068787,
  1.548920476436615,
  1.5488492131233216,
  1.5521193325519562,
  1.5398542523384093,
  1.5460204780101776,
  1.542522770166397,
  1.545697319507599,
  1.5398574352264405,
  1.5343843519687652,
  1.527653968334198,
  1.5278605341911315,
  1.5248642086982727,
  1.5329119861125946,
  1.5154

In [29]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["W_query", "W_key", "W_value", "out_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 393,216 || all params: 70,055,424 || trainable%: 0.5613


## 3. Dataset
Using DialogSum — same dataset as the Flan-T5 PEFT notebook.

In [23]:
dataset = load_dataset("knkarthick/dialogsum")
print(dataset)
print(dataset["train"][0])

d:\GptFromScratch\gpt_env\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shaha\.cache\huggingface\hub\datasets--knkarthick--dialogsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1500/1500 [00:00<00:00, 98037.46 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is

In [24]:
MAX_LENGTH = 256
START_PROMPT = "Summarize the following conversation.\n\n"
END_PROMPT = "\n\nSummary: "

def tokenize_and_mask(example):
    instruction_text = START_PROMPT + example["dialogue"] + END_PROMPT
    response_text = example["summary"] + tokenizer.eos_token
    full_text = instruction_text + response_text

    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    instruction_ids = tokenizer.encode(instruction_text, add_special_tokens=False)
    instruction_len = len(instruction_ids)

    full_ids = full_ids[:MAX_LENGTH]

    # mask instruction tokens — loss only on summary
    labels = [-100] * min(instruction_len, len(full_ids)) + full_ids[instruction_len:]
    labels = labels[:MAX_LENGTH]

    pad_id = tokenizer.pad_token_id
    seq_len = len(full_ids)
    attention_mask = [1] * seq_len + [0] * (MAX_LENGTH - seq_len)
    full_ids = full_ids + [pad_id] * (MAX_LENGTH - seq_len)
    labels = labels + [-100] * (MAX_LENGTH - len(labels))

    return {"input_ids": full_ids, "attention_mask": attention_mask, "labels": labels}

tokenized_train = dataset["train"].map(tokenize_and_mask, remove_columns=dataset["train"].column_names)
tokenized_val = dataset["validation"].map(tokenize_and_mask, remove_columns=dataset["validation"].column_names)
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
print(f"Train: {len(tokenized_train)} | Val: {len(tokenized_val)}")

Map: 100%|██████████| 500/500 [00:02<00:00, 216.11 examples/s]

Train: 12460 | Val: 500


## 4. Zero-Shot & Few-Shot Inference (Before Fine-Tuning)
Run these before training to establish a baseline. Compare results after training.

In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)]).to(device)
    prompt_len = input_ids.shape[1]
    for _ in range(max_new_tokens):
        input_ids_cond = input_ids[:, -512:]
        with torch.no_grad():
            logits = model(input_ids_cond).logits[:, -1, :] / temperature
        top_vals, _ = torch.topk(logits, top_k)
        logits[logits < top_vals[:, -1:]] = float("-inf")
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        if next_id.item() == tokenizer.eos_token_id:
            break
        input_ids = torch.cat([input_ids, next_id], dim=1)
    return tokenizer.decode(input_ids[0][prompt_len:].tolist())

In [31]:
idx = 100
dialogue = dataset["test"][idx]["dialogue"]
reference = dataset["test"][idx]["summary"]

zero_shot_prompt = START_PROMPT + dialogue + END_PROMPT
zero_shot_output = generate_text(zero_shot_prompt)

dash = "-" * 80
print(dash)
print(f"DIALOGUE:\n{dialogue}")
print(dash)
print(f"HUMAN SUMMARY:\n{reference}")
print(dash)
print(f"ZERO-SHOT OUTPUT:\n{zero_shot_output}")

--------------------------------------------------------------------------------
DIALOGUE:
#Person1#: OK, that's a cut! Let's start from the beginning, everyone.
#Person2#: What was the problem that time?
#Person1#: The feeling was all wrong, Mike. She is telling you that she doesn't want to see you any more, but I want to get more anger from you. You're acting hurt and sad, but that's not how your character would act in this situation.
#Person2#: But Jason and Laura have been together for three years. Don't you think his reaction would be one of both anger and sadness?
#Person1#: At this point, no. I think he would react the way most guys would, and then later on, we would see his real feelings.
#Person2#: I'm not so sure about that.
#Person1#: Let's try it my way, and you can see how you feel when you're saying your lines. After that, if it still doesn't feel right, we can try something else.
--------------------------------------------------------------------------------
HUMAN SUMMA

In [32]:
# Few-shot: include 2 (dialogue, summary) examples before the actual query
def build_few_shot_prompt(dialogue, n_shots=2):
    prompt = ""
    for i in range(n_shots):
        ex = dataset["train"][i]
        prompt += START_PROMPT + ex["dialogue"] + END_PROMPT + ex["summary"] + "\n\n"
    prompt += START_PROMPT + dialogue + END_PROMPT
    return prompt

few_shot_prompt = build_few_shot_prompt(dialogue, n_shots=2)
few_shot_output = generate_text(few_shot_prompt)

print(dash)
print(f"HUMAN SUMMARY:\n{reference}")
print(dash)
print(f"FEW-SHOT OUTPUT (2-shot):\n{few_shot_output}")

--------------------------------------------------------------------------------
HUMAN SUMMARY:
#Person1# and Mike have a disagreement on how to act out a scene. #Person1# proposes that Mike can try to act in #Person1#'s way.
--------------------------------------------------------------------------------
FEW-SHOT OUTPUT (2-shot):
Yes, let's do it!"

And so they went on a different adventure and were both happy. They looked forward to coming to the next point to see what it would be like to be friends again.
Once upon a time there was an ordinary family. They lived in a big house and were very happy together. One day, the family was playing together when they heard a loud noise outside.

The family was curious, so they opened the window to look outside. It was a big truck with a lot of boxes. The truck was


## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir="checkpoints/lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    warmup_steps=100,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="epoch",
    logging_steps=50,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

trainer.train()

In [ ]:
model.save_pretrained("checkpoints/lora/final")
print("LoRA adapters saved.")

## 5. ROUGE Evaluation


In [ ]:
def generate_summary(dialogue, max_new_tokens=100, temperature=0.8, top_k=40):
    model.eval()
    device = next(model.parameters()).device
    prompt = START_PROMPT + dialogue + END_PROMPT
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    input_ids = torch.tensor([prompt_ids]).to(device)
    for _ in range(max_new_tokens):
        input_ids_cond = input_ids[:, -512:]
        with torch.no_grad():
            logits = model(input_ids_cond).logits[:, -1, :] / temperature
        top_vals, _ = torch.topk(logits, top_k)
        logits[logits < top_vals[:, -1:]] = float("-inf")
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        if next_id.item() == tokenizer.eos_token_id:
            break
        input_ids = torch.cat([input_ids, next_id], dim=1)
    return tokenizer.decode(input_ids[0][len(prompt_ids):].tolist())

# quick sanity check on one example
idx = 200
dialogue = dataset["test"][idx]["dialogue"]
reference = dataset["test"][idx]["summary"]
generated = generate_summary(dialogue)

dash = "-" * 80
print(dash)
print(f"HUMAN SUMMARY:\n{reference}")
print(dash)
print(f"MODEL SUMMARY:\n{generated}")

In [ ]:
!pip install nltk rouge_score -q

rouge = evaluate.load("rouge")

# use 200 samples for speed
dialogues = dataset["test"]["dialogue"][:200]
human_summaries = dataset["test"]["summary"][:200]

# --- Original model (LoRA disabled) ---
model.disable_adapter_layers()
original_summaries = []
for dialogue in dialogues:
    original_summaries.append(generate_summary(dialogue))
model.enable_adapter_layers()

# --- PEFT model (LoRA enabled) ---
peft_summaries = []
for dialogue in dialogues:
    peft_summaries.append(generate_summary(dialogue))

# --- ROUGE ---
original_results = rouge.compute(
    predictions=original_summaries,
    references=list(human_summaries),
    use_aggregator=True,
    use_stemmer=True,
)
peft_results = rouge.compute(
    predictions=peft_summaries,
    references=list(human_summaries),
    use_aggregator=True,
    use_stemmer=True,
)

dash = "-" * 60
print(dash)
print(f"{'METRIC':<12} {'ORIGINAL':>12} {'PEFT':>12} {'DELTA':>12}")
print(dash)
for k in original_results:
    orig = original_results[k]
    peft = peft_results[k]
    delta = peft - orig
    print(f"{k:<12} {orig:>12.4f} {peft:>12.4f} {delta:>+12.4f}")
print(dash)